# Graded Lab: Agentic Workflows

In this lab, you will build an agentic system that generates a short research report through planning, external tool usage, and feedback integration. Your workflow will involve:

### Agents

* **Planning Agent / Writer**: Creates an outline and coordinates tasks.
* **Research Agent**: Gathers external information using tools like Arxiv, Tavily, and Wikipedia.
* **Editor Agent**: Reflects on the report and provides suggestions for improvement.

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to write your solution code or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the <span style="background-color: red; color: white; padding: 3px 5px; font-size: 16px; border-radius: 5px;">Submit assignment</span> button on the top right of the page.
---


### Research Tools

By importing `research_tools`, you gain access to several search utilities:

- `research_tools.arxiv_search_tool(query)` → search academic papers from **arXiv**  

  *Example:* `research_tools.arxiv_search_tool("neural networks for climate modeling")`

- `research_tools.tavily_search_tool(query)` → perform web searches with the **Tavily API**  

  *Example:* `research_tools.tavily_search_tool("latest trends in sunglasses fashion")`

- `research_tools.wikipedia_search_tool(query)` → retrieve summaries from **Wikipedia**  

  *Example:* `research_tools.wikipedia_search_tool("Ensemble Kalman Filter")`

Run the cell below to make them available.

In [1]:
# =========================
# Imports
# =========================

# --- Standard library 
from datetime import datetime
import re
import json
import ast


# --- Third-party ---
from IPython.display import Markdown, display
from aisuite import Client

# --- Local / project ---
import research_tools

In [2]:
import unittests

### Initialize client

Create a shared client instance for upcoming calls.

In [3]:
CLIENT = Client()

## Exercise 1: planner_agent

### Objective
Correctly set up a call to a language model (LLM) to generate a research plan.

### Instructions

1. **Focus Areas**:
   - Ensure `CLIENT.chat.completions.create` is correctly configured.
   - Pass the `model` and `messages` parameters correctly:
     - **Model**: Use `"openai:o4-mini"` by default.
     - **Messages**: Set with `{"role": "user", "content": user_prompt}`.
     - **Temperature**: Fixed at 1 for creative outputs.

### Notes

- The prompt is pre-defined and guides the LLM on task requirements.
- Only return a formatted list of steps — no extra text.

Focus on the LLM call setup to complete the task.

In [4]:
# GRADED FUNCTION: planner_agent

def planner_agent(topic: str, model: str = "openai:o4-mini") -> list[str]:
    """
    Generates a plan as a Python list of steps (strings) for a research workflow.

    Args:
        topic (str): Research topic to investigate.
        model (str): Language model to use.

    Returns:
        List[str]: A list of executable step strings.
    """

    
    # Build the user prompt
    user_prompt = f"""
    You are a planning agent responsible for organizing a research workflow with multiple intelligent agents.

    🧠 Available agents:
    - A research agent who can search the web, Wikipedia, and arXiv.
    - A writer agent who can draft research summaries.
    - An editor agent who can reflect and revise the drafts.

    🎯 Your job is to write a clear, step-by-step research plan **as a valid Python list**, where each step is a string.
    Each step should be atomic, executable, and must rely only on the capabilities of the above agents.

    🚫 DO NOT include irrelevant tasks like "create CSV", "set up a repo", "install packages", etc.
    ✅ DO include real research-related tasks (e.g., search, summarize, draft, revise).
    ✅ DO assume tool use is available.
    ✅ DO NOT include explanation text — return ONLY the Python list.
    ✅ The final step should be to generate a Markdown document containing the complete research report.

    Topic: "{topic}"
    """

    # Add the user prompt to the messages list
    messages = [{"role": "user", "content": user_prompt}]

    ### START CODE HERE ###

    # Call the LLM
    response = CLIENT.chat.completions.create( 
        # Pass in the model
        model=model,
        # Define the messages. Remember this is meant to be a user prompt!
        messages=messages,
        # Keep responses creative
        temperature=1, 
    )

    ### END CODE HERE ###

    # Extract message from response
    steps_str = response.choices[0].message.content.strip()

    # Parse steps
    steps = ast.literal_eval(steps_str)

    return steps

In [5]:
# Test your code!
unittests.test_planner_agent(planner_agent)

 All tests passed!


## Exercise 2: research_agent

### Objective
Set up a call to a language model (LLM) to perform a research task using various tools.

### Instructions

**Focus Areas**:

- **Creating a Custom Prompt**:
  - **Define the Role**: Clearly specify the role, such as "research assistant."
  - **List Available Tools** (as strings inside the prompt, not the actual functions):
    - Use `arxiv_tool` to find academic papers.
    - Use `tavily_tool` for general web searches.
    - Use `wikipedia_tool` for accessing encyclopedic knowledge.
  - **Specify the Task**: Include a placeholder in your prompt for defining the specific task that needs to be accomplished.
  - **Include Date Information**: Add a placeholder for the current date or time to provide context.

- **Creating Messages Dict**:
  - Ensure the `messages` are correctly set with `{"role": "user", "content": prompt}`.

- **Creating Tools List**:
  - Create a list of tools for use, such as `research_tools.arxiv_search_tool`, `research_tools.tavily_search_tool`, and `research_tools.wikipedia_search_tool`.

- **Correctly Setting the Call to the LLM**:
  - Pass the `model`, `messages`, and `tools` parameters accurately.
  - Set `tool_choice` to `"auto"` for automatic tool selection.
  - Limit interactions with `max_turns=6`.

### Notes

- The function provides pre-coded blocks where you need to replace placeholder values.
- The approach allows the LLM to use tools dynamically based on the task.

Focus on accurately setting the messages, tools, and LLM call parameters to complete the task.

In [22]:
print(dir(research_tools))

['ET', 'TavilyClient', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'arxiv_search_tool', 'arxiv_tool_def', 'load_dotenv', 'os', 'requests', 'session', 'tavily_search_tool', 'tavily_tool_def', 'tool_mapping', 'wikipedia', 'wikipedia_search_tool', 'wikipedia_tool_def']


In [23]:
# GRADED FUNCTION: research_agent

def research_agent(task: str, model: str = "openai:gpt-4o", return_messages: bool = False):
    """
    Executes a research task using tools via aisuite (no manual loop).
    Returns either the assistant text, or (text, messages) if return_messages=True.
    """
    print("==================================")  
    print("🔍 Research Agent")                 
    print("==================================")

    current_time = datetime.now().strftime('%Y-%m-%d')
    
    ### START CODE HERE ###

    # Create a customizable prompt by defining the role (e.g., "research assistant"),
    # listing tools (arxiv_tool, tavily_tool, wikipedia_tool) for various searches,
    # specifying the task with a placeholder, and including a current_time placeholder.
    prompt = f"""
    You are a research assistant.
    You are equipped with various tools to conduct searches on various platforms - arxiv, tavily and wikipedia.
    Given the task, accomplish it with the given tools.
    Task given: {task}
    Current date: {current_time}
    """
    
    # Create the messages dict to pass to the LLM. Remember this is a user prompt!
    messages = [{"role": "user", "content": prompt}]

    # Save all of your available tools in the tools list. These can be found in the research_tools module.
    # You can identify each tool in your list like this: 
    # research_tools.<name_of_tool>, where <name_of_tool> is replaced with the function name of the tool.
    tools = [research_tools.arxiv_search_tool , research_tools.tavily_search_tool , research_tools.wikipedia_search_tool ]
    
    # Call the model with tools enabled
    response = CLIENT.chat.completions.create(  
        # Set the model
        model=model,
        # Pass in the messages. You already defined this!
        messages=messages,
        # Pass in the tools list. You already defined this!
        tools=tools,
        # Set the LLM to automatically choose the tools
        tool_choice="auto",
        # Set the max turns to 6
        max_turns=6
    )  
    
    ### END CODE HERE ###

    content = response.choices[0].message.content
    print("✅ Output:\n", content)

    
    return (content, messages) if return_messages else content  

In [24]:
# Test your code!
unittests.test_research_agent(research_agent)

🔍 Research Agent
✅ Output:
 Here are three key references along with brief summaries:

1. **Efficient Password-Typed Key Agreement Scheme ([Paper](http://arxiv.org/abs/1002.1163v1), [PDF](https://arxiv.org/pdf/1002.1163v1))** by Sattar J Aboud
   - This paper examines a verifier password-typed key agreement scheme by Lee, Kim, and Yoo. It highlights flaws in the existing scheme and proposes an improved version that resists password guessing attacks and stolen verifier attacks. The proposed scheme is claimed to be more secure and efficient than its predecessor.

2. **Integrating HCI Datasets in Project-Based Machine Learning Courses: A College-Level Review and Case Study ([Paper](http://arxiv.org/abs/2408.03472v1), [PDF](https://arxiv.org/pdf/2408.03472v1))** by Xiaodong Qu, Matthew Key, Eric Luo, Chuhui Qiu
   - This study investigates the use of Human-Computer Interface (HCI) datasets in machine learning courses. It identifies best practices for integrating these datasets into project

## Exercise 3: writer_agent

### Objective
Set up a call to a language model (LLM) for executing writing tasks like drafting, expanding, or summarizing text.

### Instructions

1. **Focus Areas**:
   - **System Prompt**:
     - Define `system_prompt` to assign the LLM the role of a writing agent focused on generating academic or technical content.
   - **System and User Messages**:
     - Create `system_msg` using `{"role": "system", "content": system_prompt}`.
     - Create `user_msg` using `{"role": "user", "content": task}`.
   - **Messages List**:
     - Combine `system_msg` and `user_msg` into a `messages` list.

### Notes

- The function is designed to produce well-structured text by setting the correct prompts.
- Temperature is set to 1.0 to allow for creative variance in the writing outputs.

Ensure the system prompt and messages are defined properly to achieve a structured output from the LLM.

In [25]:
# GRADED FUNCTION: writer_agent
def writer_agent(task: str, model: str = "openai:gpt-4o") -> str: # @REPLACE def writer_agent(task: str, model: str = None) -> str:
    """
    Executes writing tasks, such as drafting, expanding, or summarizing text.
    """
    print("==================================")
    print("✍️ Writer Agent")
    print("==================================")

    ### START CODE HERE ###
    
    # Create the system prompt.
    # This should assign the LLM the role of a writing agent specialized in generating well-structured academic or technical content
    system_prompt = f"""
    You are an expert writing agent specialized in generating well-structured academic or technical content.
    """

    # Define the system msg by using the system_prompt and assigning the role of system
    system_msg = system_prompt

    # Define the user msg. In this case the user prompt should be the task passed to the function
    user_msg = f"Task to be accomplish: {task}"

    # Add both system and user messages to the messages list
    messages = [
        {"role":"system", "content":system_msg},
        {"role":"user","content":user_msg}
    ]
    
    ### END CODE HERE ###

    response = CLIENT.chat.completions.create(
        model=model, 
        messages=messages,
        temperature=1.0
    )

    return response.choices[0].message.content

In [26]:
# Test your code!
unittests.test_writer_agent(writer_agent)

✍️ Writer Agent
 All tests passed!


## Exercise 4: editor_agent

### Objective
Configure a call to a language model (LLM) to perform editorial tasks such as reflecting, critiquing, or revising drafts.

### Instructions

1. **Focus Areas**:
   - **System Prompt**:
     - Define `system_prompt` to assign the LLM the role of an editor agent whose task is to reflect on, critique, or improve drafts.
   - **System and User Messages**:
     - Create `system_msg` using `{"role": "system", "content": system_prompt}`.
     - Create `user_msg` using `{"role": "user", "content": task}`.
   - **Messages List**:
     - Combine `system_msg` and `user_msg` into a `messages` list.

### Notes

- The editor agent is tailored for enhancing the quality of text by setting an appropriate role and task in the prompts.
- Temperature is set to 0.7, balancing creativity and coherence in editorial outputs.

Ensure the system prompt and messages are accurately set up to perform effective editorial tasks with the LLM.

In [27]:
# GRADED FUNCTION: editor_agent
def editor_agent(task: str, model: str = "openai:gpt-4o") -> str:
    """
    Executes editorial tasks such as reflection, critique, or revision.
    """
    print("==================================")
    print("🧠 Editor Agent")
    print("==================================")
    
    ### START CODE HERE ###

    # Create the system prompt.
    # This should assign the LLM the role of an editor agent specialized in reflecting on, critiquing, or improving existing drafts.
    system_prompt = f"""
    You are an expert editor agent specialized in reflecting on, critiquing, or improving existing drafts.
    """
    
    # Define the system msg by using the system_prompt and assigning the role of system
    system_msg = system_prompt
    
    # Define the user msg. In this case the user prompt should be the task passed to the function
    user_msg = f"Task to be accomplished: {task}"
    
    # Add both system and user messages to the messages list
    messages = [
        {"role":"system", "content": system_msg},
        {"role":"user", "content": user_msg}
    ]
    
    ### END CODE HERE ###
    
    response = CLIENT.chat.completions.create(
        model=model, 
        messages=messages,
        temperature=0.7 
    )
    
    return response.choices[0].message.content

In [28]:
# Test your code!
unittests.test_editor_agent(editor_agent)

🧠 Editor Agent
 All tests passed!


### 🎯 The Executor Agent

The `executor_agent` manages the workflow by executing each step of a given plan. It:

1. Decides **which agent** (`research_agent`, `writer_agent`, or `editor_agent`) should handle the step.
2. Builds context from the outputs of previous steps.
3. Sends the enriched task to the selected agent.
4. Collects and stores the results in a shared history.

👉 **Do not implement or modify this function.** It is already provided as the orchestration component of the multi-agent pipeline.

Notice that `planner_agent` might return a long list of steps. Because of this, the maximum number of steps is set to a maximum of 4 to keep running time reasonable.

In [29]:
agent_registry = {
    "research_agent": research_agent,
    "editor_agent": editor_agent,
    "writer_agent": writer_agent,
}

def clean_json_block(raw: str) -> str:
    """
    Clean the contents of a JSON block that may come wrapped with Markdown backticks.
    """
    raw = raw.strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```(?:json)?\n?", "", raw)
        raw = re.sub(r"\n?```$", "", raw)
    return raw.strip()

In [30]:
def executor_agent(topic, model: str = "openai:gpt-4o", limit_steps: bool = True):

    plan_steps = planner_agent(topic)
    max_steps = 4

    if limit_steps:
        plan_steps = plan_steps[:min(len(plan_steps), max_steps)]
    
    history = []

    print("==================================")
    print("🎯 Editor Agent")
    print("==================================")

    for i, step in enumerate(plan_steps):

        agent_decision_prompt = f"""
        You are an execution manager for a multi-agent research team.

        Given the following instruction, identify which agent should perform it and extract the clean task.

        Return only a valid JSON object with two keys:
        - "agent": one of ["research_agent", "editor_agent", "writer_agent"]
        - "task": a string with the instruction that the agent should follow

        Only respond with a valid JSON object. Do not include explanations or markdown formatting.

        Instruction: "{step}"
        """
        response = CLIENT.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": agent_decision_prompt}],
            temperature=0,
        )

        raw_content = response.choices[0].message.content
        cleaned_json = clean_json_block(raw_content)
        agent_info = json.loads(cleaned_json)

        agent_name = agent_info["agent"]
        task = agent_info["task"]

        context = "\n".join([
            f"Step {j+1} executed by {a}:\n{r}" 
            for j, (s, a, r) in enumerate(history)
        ])
        enriched_task = f"""
        You are {agent_name}.

        Here is the context of what has been done so far:
        {context}

        Your next task is:
        {task}
        """

        print(f"\n🛠️ Executing with agent: `{agent_name}` on task: {task}")

        if agent_name in agent_registry:
            output = agent_registry[agent_name](enriched_task)
            history.append((step, agent_name, output))
        else:
            output = f"⚠️ Unknown agent: {agent_name}"
            history.append((step, agent_name, output))

        print(f"✅ Output:\n{output}")

    return history

In [31]:
# If you want to see the full workflow without limiting the number of steps. Set limit_steps to False
# Keep in mind this could take more than 10 minutes to complete
executor_history = executor_agent("The ensemble Kalman filter for time series forecasting", limit_steps=True)

md = executor_history[-1][-1].strip("`")  
display(Markdown(md))

🎯 Editor Agent

🛠️ Executing with agent: `research_agent` on task: search web and Wikipedia for theory and definitions of the ensemble Kalman filter
🔍 Research Agent
✅ Output:
 Here is what I found regarding the Ensemble Kalman Filter from both Tavily and Wikipedia:

### Tavily Search Results:
1. **Aspects of the Ensemble Kalman Filter** ([PDF Link](https://www.reading.ac.uk/maths-and-stats/-/media/project/uor-main/schools-departments/maths/documents/livingspdf.pdf?la=en&hash=E3DF5251538E38D9B1A107AF10157473))
   - The Ensemble Kalman Filter (EnKF) is designed to provide state estimates by blending model information with observations. Compared to other methods, it offers computational advantages and represents nonlinearity with an ensemble of state estimates rather than a single estimate.
   
2. **The Ensemble Kalman filter: A Signal Processing Perspective** ([Link](https://link.springer.com/article/10.1186/s13634-017-0492-x))
   - EnKF is a Monte Carlo-based implementation suitable fo


🛠️ Executing with agent: `research_agent` on task: compile a bibliography of the most relevant publications on ensemble Kalman filter for time series forecasting
🔍 Research Agent
✅ Output:
 Here's a compilation of relevant publications on the Ensemble Kalman Filter for time series forecasting:

1. **Ensemble Kalman Filtering Meets Gaussian Process SSM for Non-Mean-Field and Online Inference**  
   - Authors: Zhidi Lin, Yiyong Sun, Feng Yin, Alexandre Hoang Thiéry  
   - Published: 2023-12-10  
   - Summary: This paper integrates the Ensemble Kalman Filter (EnKF) with Gaussian Process state-space models for online learning, focusing on improving inference accuracy and parameterization. It offers a detailed analysis and insights into the performance of EnKF-aided variational inference algorithms.  
   - [Read more](https://arxiv.org/abs/2312.05910v5), [PDF](https://arxiv.org/pdf/2312.05910v5)

2. **LLM-Mixer: Multiscale Mixing in LLMs for Time Series Forecasting**  
   - Authors: Md Kow

Here's a compilation of relevant publications on the Ensemble Kalman Filter for time series forecasting:

1. **Ensemble Kalman Filtering Meets Gaussian Process SSM for Non-Mean-Field and Online Inference**  
   - Authors: Zhidi Lin, Yiyong Sun, Feng Yin, Alexandre Hoang Thiéry  
   - Published: 2023-12-10  
   - Summary: This paper integrates the Ensemble Kalman Filter (EnKF) with Gaussian Process state-space models for online learning, focusing on improving inference accuracy and parameterization. It offers a detailed analysis and insights into the performance of EnKF-aided variational inference algorithms.  
   - [Read more](https://arxiv.org/abs/2312.05910v5), [PDF](https://arxiv.org/pdf/2312.05910v5)

2. **LLM-Mixer: Multiscale Mixing in LLMs for Time Series Forecasting**  
   - Authors: Md Kowsher, Md. Shohanur Islam Sobuj, Nusrat Jahan Prottasha, E. Alejandro Alanis, Ozlem Ozmen Garibay, Niloofar Yousefi  
   - Published: 2024-10-15  
   - Summary: LLM-Mixer combines multiscale time-series decomposition with pre-trained language models to enhance forecasting accuracy. It offers insights into leveraging multiscale analysis and language models for scalable time-series forecasting.  
   - [Read more](https://arxiv.org/abs/2410.11674v2), [PDF](https://arxiv.org/pdf/2410.11674v2)

3. **Forecasting Intermittent Time Series with Gaussian Processes and Tweedie Likelihood**  
   - Authors: Stefano Damato, Dario Azzimonti, Giorgio Corani  
   - Published: 2025-02-26  
   - Summary: This paper uses Gaussian Processes for probabilistic forecasting of intermittent time series, focusing on improved flexibility and accuracy over traditional methods.  
   - [Read more](https://arxiv.org/abs/2502.19086v5), [PDF](https://arxiv.org/pdf/2502.19086v5)

4. **The Geometric Unscented Kalman Filter**  
   - Authors: Chengling Fang, Jiang Liu, Songqing Ye, Ju Zhang  
   - Published: 2020-09-28  
   - Summary: Introduces a new nonlinear estimation scheme called the Geometric Unscented Kalman Filter (GUF), which maintains high accuracy and stability in estimation tasks.  
   - [Read more](https://arxiv.org/abs/2009.13079v1), [PDF](https://arxiv.org/pdf/2009.13079v1)

5. **Probabilistic Hierarchical Forecasting with Deep Poisson Mixtures**  
   - Authors: Kin G. Olivares, O. Nganba Meetei, Ruijun Ma, Rohan Reddy, Mengfei Cao, Lee Dicker  
   - Published: 2021-10-25  
   - Summary: This work addresses hierarchical forecasting challenges using a combination of neural networks and statistical models to maintain coherence and accuracy in probabilistic forecasts.  
   - [Read more](https://arxiv.org/abs/2110.13179v8), [PDF](https://arxiv.org/pdf/2110.13179v8)

These publications contain detailed algorithms, theoretical insights, and more on the application of Ensemble Kalman Filters in time series forecasting.

## Check grading feedback

If you have collapsed the right panel to have more screen space for your code, as shown below:

<img src="./images/collapsed.png" alt="Collapsed Image" width="800" height="400"/>

You can click on the left-facing arrow button (highlighted in red) to view feedback for your submission after submitting it for grading. Once expanded, it should display like this:

<img src="./images/expanded.png" alt="Expanded Image" width="800" height="400"/>